# Descarga de imágenes - Ejemplo práctico

Objetivo:
- Leer el catálogo oficial de cámaras.
- Descargar una imagen de cada cámara desde `URLIMAGEN`.
- Almacenar las capturas en formato reproducible (`raw/<camara>/<YYYYMMDDTHHMMSSZ>.jpg`, hora UTC).
- Generar un fichero de metadatos.

La captura continua de las cámaras seleccionadas la realiza el servicio `scripts/captura_camaras.py`; este notebook sirve para obtener una instantánea de **todas** las cámaras del catálogo.

In [1]:
import logging
import sys
from datetime import datetime, timezone
from pathlib import Path
from time import sleep

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from catalogo import load_catalog
from config import load_config, path
from tiempo import FILENAME_FORMAT

CFG = load_config()
RAW_DIR = path("raw")
METADATA_DIR = path("metadata")
RAW_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT = CFG["captura"]["timeout_s"]
SLEEP_SECONDS = 1

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

In [2]:
# El catálogo ya limpia URLIMAGEN (algunas traen un tabulador final que provocaba un 404)
df = load_catalog()
print(f"Cámaras encontradas: {len(df)}")

required_columns = ["ID", "NOMBRE", "URLIMAGEN"]
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Falta columna {col}")
print("Validación correcta")

df.head()

Cámaras encontradas: 134
Validación correcta


,FID,ID,NOMBRE,DESCRIPCION,DIRECCION,SDOGEOMETRIA,URL,EMAIL,CONTACTO,HORARIOS,PRECIOS,TITULARIDAD,TARJETAJOVEN,ACCESOPMR,URLIMAGEN,lat,lon
0,da_camarasTrafico.fid--3b3f458b_19b968d046b_cbe,21611,TV87-EL BONI - CAMPANILLAS,C/ José Calderón - El Boni,"CALLE JOSÉ CALDERÓN, 45",POINT (-4.539062997820292 36.72406291352086),https://movilidad.malaga.eu/es/servicios/camar...,NaN,NaN,NaN,NaN,MUNICIPAL,No,No,https://ctraficomovilidad.malaga.eu/recursos/m...,36.724063,-4.539063
1,da_camarasTrafico.fid--3b3f458b_19b968d046b_cbf,75066,TV100-JOSÉ CALDERÓN - JUNTA DE DISTRITO CAMPAN...,José Calderón - Junta De Distrito Campanillas,"CALLE JOSÉ CALDERÓN, 174",POINT (-4.544862290941862 36.72806421666882),https://movilidad.malaga.eu/es/servicios/camar...,NaN,NaN,NaN,NaN,MUNICIPAL,No,No,https://ctraficomovilidad.malaga.eu/recursos/m...,36.728064,-4.544862
2,da_camarasTrafico.fid--3b3f458b_19b968d046b_cc0,75086,TV120-HERMANOS LUMIERE - LICURGO,Hermanos Lumiere - Licurgo,"CALLE HERMANOS LUMIÈRE, 14",POINT (-4.4627101617254334 36.69370759993173),https://movilidad.malaga.eu/es/servicios/camar...,NaN,NaN,NaN,NaN,MUNICIPAL,No,No,https://ctraficomovilidad.malaga.eu/recursos/m...,36.693708,-4.462710
3,da_camarasTrafico.fid--3b3f458b_19b968d046b_cc1,75093,TV127-CARRIL DEL CAPITÁN - JUAN DE ROBLES,Carril Del Capitán - Juan De Robles,"CALLE JUAN DE ROBLES, 55",POINT (-4.46161103014071 36.721554097509305),https://movilidad.malaga.eu/es/servicios/camar...,NaN,NaN,NaN,NaN,MUNICIPAL,No,No,https://ctraficomovilidad.malaga.eu/recursos/m...,36.721554,-4.461611
4,da_camarasTrafico.fid--3b3f458b_19b968d046b_cc2,75069,TV103-AVDA LOS GUINDOS - CMNO DE LA TÉRMICA,Avda Los Guindos - Cmno De La Térmica,"AVENIDA DE LOS GUINDOS, 44",POINT (-4.44701655450928 36.69082158314863),https://movilidad.malaga.eu/es/servicios/camar...,NaN,NaN,NaN,NaN,MUNICIPAL,No,No,https://ctraficomovilidad.malaga.eu/recursos/m...,36.690822,-4.447017


In [3]:
# Sesión global con reintentos y espera exponencial (1 s, 2 s, 4 s...)
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

# Cabeceras indispensables para evitar el error 403 Forbidden
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://malaga.eu",
    "Accept": "image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
}


def download_image(image_url, output_file):
    response = session.get(image_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    output_file.write_bytes(response.content)


def capture_camera(row):
    camera_id = row["ID"]
    camera_name = str(row["NOMBRE"])
    image_url = row["URLIMAGEN"]

    if pd.isna(image_url):
        return None

    # Nombre de fichero en UTC, igual que captura_camaras.py
    timestamp = datetime.now(timezone.utc).strftime(FILENAME_FORMAT)
    camera_dir = RAW_DIR / camera_id
    camera_dir.mkdir(exist_ok=True)
    image_file = camera_dir / f"{timestamp}.jpg"

    try:
        download_image(image_url, image_file)
        status, saved = "OK", image_file.relative_to(path("datos")).as_posix()
    except Exception as e:
        logger.error(f"{camera_name}: {e}")
        status, saved = f"ERROR: {e}", None

    return {
        "camera_id": camera_id,
        "camera_name": camera_name,
        "image_url": image_url,
        "capture_time": timestamp,
        "image_file": saved,
        "status": status,
    }

In [4]:
results = []
for _, row in df.iterrows():
    result = capture_camera(row)
    if result:
        results.append(result)
    sleep(SLEEP_SECONDS)

captures_df = pd.DataFrame(results)
captures_df["status"].str.startswith("OK").value_counts()

status
True    134
Name: count, dtype: int64

In [5]:
metadata_file = METADATA_DIR / "captura_imagenes.csv"
captures_df.to_csv(metadata_file, index=False)
print(f"Metadatos guardados en: {metadata_file}")

captures_df.sample(min(10, len(captures_df)))

Metadatos guardados en: C:\Users\loren\OneDrive\Escritorio\MASTER-UIMP\TFM\tfm\data\malaga\metadata\captura_imagenes.csv


,camera_id,camera_name,image_url,capture_time,image_file,status
78,75079,TV114-AVDA AURORA - CONDE DE GUADALHORCE,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230558Z,raw/75079/20261012T230558Z.jpg,OK
12,21546,TV07-BOQUETE DEL MUELLE,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230444Z,raw/21546/20261012T230444Z.jpg,OK
115,21583,TV59-NAVARRO LEDESMA,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230640Z,raw/21583/20261012T230640Z.jpg,OK
76,21590,TV66-CMNO SAN RAFAEL,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230556Z,raw/21590/20261012T230556Z.jpg,OK
30,21613,TV89-JOAQUÍN VARGAS,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230503Z,raw/21613/20261012T230503Z.jpg,OK
66,75087,TV121-MARTÍNEZ DE LA ROSA - VANDELVIRA,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230546Z,raw/75087/20261012T230546Z.jpg,OK
96,21605,TV81-ATARAZANAS,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230618Z,raw/21605/20261012T230618Z.jpg,OK
70,21550,TV11-EUGENIO GROSS,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230550Z,raw/21550/20261012T230550Z.jpg,OK
101,21538,TV22-CARLOS HAYA,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230623Z,raw/21538/20261012T230623Z.jpg,OK
34,21567,TV43-DOS ACERAS,https://ctraficomovilidad.malaga.eu/recursos/m...,20261012T230507Z,raw/21567/20261012T230507Z.jpg,OK
